# Executive Summary Generator

Queries the mart tables for the most recent complete month's KPIs,
compares them to the prior three months, and uses the Claude API to
generate a plain-English executive summary suitable for a leadership audience.

**Sections**
1. Setup — BigQuery client and Anthropic client
2. KPI pull — revenue, orders, AOV, return rate, margin from `fct_orders`
3. Customer mix — new vs. repeat split
4. Traffic source — session volume from `fct_funnel`
5. Build prompt — assemble KPI context
6. Generate summary — Claude API call
7. Display output

## 1. Setup

In [1]:
import sys
sys.path.append('..')
import config

import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
from datetime import datetime
from google.cloud import bigquery
import anthropic

# Load .env from the project root (one level up from notebooks/)
load_dotenv(Path('..') / '.env')

BQ_PROJECT = config.PROJECT_ID
DATASET    = 'ecomm_marts'

bq_client      = bigquery.Client(project=BQ_PROJECT)
claude_client  = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from .env
CLAUDE_MODEL   = 'claude-haiku-4-5-20251001'

def q(sql):
    """Run a BigQuery SQL string and return a DataFrame."""
    return bq_client.query(sql).to_dataframe()

print(f'BQ project  : {BQ_PROJECT}')
print(f'Claude model: {CLAUDE_MODEL}')
print(f'Timestamp   : {datetime.now().strftime("%Y-%m-%d %H:%M")}')

Credentials: /Users/marcalexander/projects/ai_orchestrator_claude/portfolio_ecomm/notebooks/../credentials/credentials.json
Project ID:  portfolio-thelook
BQ project  : portfolio-thelook
Claude model: claude-haiku-4-5-20251001
Timestamp   : 2026-04-06 11:48


## 2. KPI Pull — Revenue, Orders, AOV, Return Rate, Margin

Aggregates `fct_orders` to the monthly grain for the last 4 complete months.
The most recent month becomes the **reporting period**; the prior three form
the baseline for MoM comparisons.

In [2]:
sql_kpis = f"""
select
    format_date('%Y-%m', date(created_at))          as month,
    count(distinct order_id)                         as orders,
    round(sum(revenue_amt), 2)                       as revenue,
    round(avg(revenue_amt), 2)                       as aov,
    round(sum(margin_amt), 2)                        as margin,
    round(avg(margin_amt / nullif(revenue_amt, 0)) * 100, 1) as margin_pct,
    round(avg(returned_ind) * 100, 1)                as return_rate
from `{BQ_PROJECT}.{DATASET}.fct_orders`
where completed_ind = 1
  and date(created_at) < date_trunc(current_date(), month)   -- exclude current partial month
  and date(created_at) >= date_sub(
        date_trunc(current_date(), month), interval 4 month
      )
group by 1
order by 1 desc
limit 4
"""

kpis = q(sql_kpis)
kpis

/opt/anaconda3/envs/testing/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,orders,revenue,aov,margin,margin_pct,return_rate
0,2026-03,2354,204277.88,86.78,105931.66,51.2,0.0
1,2026-02,1331,116370.79,87.43,60559.43,51.3,0.0
2,2026-01,1238,106309.90,85.87,55076.39,51.3,0.0
3,2025-12,1184,100207.59,84.63,52076.65,51.1,0.0


In [ ]:
# Split into current month vs. prior 3-month average
current   = kpis.iloc[0]                          # most recent complete month
prior_avg = kpis.iloc[1:].mean(numeric_only=True) # average of prior 3 months

def pct_change(curr, prev):
    """Return a signed percentage change string, e.g. '+48.2%'."""
    if prev == 0:
        return 'N/A'
    pct = (curr - prev) / prev * 100
    sign = '+' if pct >= 0 else ''
    return f'{sign}{pct:.1f}%'

print(f"Reporting month : {current['month']}")
print(f"Revenue         : ${current['revenue']:,.0f}  ({pct_change(current['revenue'], prior_avg['revenue'])} vs prior 3-mo avg)")
print(f"Orders          : {current['orders']:,.0f}  ({pct_change(current['orders'], prior_avg['orders'])} vs prior 3-mo avg)")
print(f"AOV             : ${current['aov']:,.2f}  ({pct_change(current['aov'], prior_avg['aov'])} vs prior 3-mo avg)")
print(f"Margin %        : {current['margin_pct']:.1f}%  ({pct_change(current['margin_pct'], prior_avg['margin_pct'])} vs prior 3-mo avg)")
print(f"Return rate     : {current['return_rate']:.1f}%  ({pct_change(current['return_rate'], prior_avg['return_rate'])} vs prior 3-mo avg)")

## 3. Customer Mix — New vs. Repeat

In [4]:
sql_cx_mix = f"""
select
    cx_order_type,
    count(distinct order_id) as orders,
    round(sum(revenue_amt), 2) as revenue
from `{BQ_PROJECT}.{DATASET}.fct_orders`
where completed_ind = 1
  and format_date('%Y-%m', date(created_at)) = '{current['month']}'
group by 1
order by 2 desc
"""

cx_mix = q(sql_cx_mix)
cx_mix

/opt/anaconda3/envs/testing/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,cx_order_type,orders,revenue
0,n/a,1295,112801.72
1,Repeat,1059,91476.16


## 4. Traffic Source — Session Volume

In [6]:
sql_traffic = f"""
select
    traffic_source,
    count(*) as sessions,
    round(avg(converted_ind) * 100, 1) as conversion_rate
from `{BQ_PROJECT}.{DATASET}.fct_funnel`
where format_date('%Y-%m', date(session_start_at)) = '{current['month']}'
group by 1
order by 2 desc
"""

traffic = q(sql_traffic)
traffic

/opt/anaconda3/envs/testing/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,traffic_source,sessions,conversion_rate
0,Email,8620,69.5
1,Adwords,5832,70.8
2,Facebook,2023,67.9
3,YouTube,1954,70.8
4,Organic,926,70.8


## 5. Build the Prompt

Assembles all KPI data into a structured context block that Claude
will use to write the executive summary.

In [8]:
# --- format customer mix for the prompt ---
cx_lines = '\n'.join(
    f"  {row['cx_order_type']}: {row['orders']:,.0f} orders, ${row['revenue']:,.0f} revenue"
    for _, row in cx_mix.iterrows()
)

# --- format traffic source for the prompt ---
traffic_lines = '\n'.join(
    f"  {row['traffic_source']}: {row['sessions']:,.0f} sessions, {row['conversion_rate']:.1f}% conversion"
    for _, row in traffic.iterrows()
)

prompt = f"""You are a senior data analyst writing a monthly executive summary for a retail e-commerce business.
Write exactly three paragraphs. Do not use bullet points or headers — prose only.

Paragraph 1 — Top-line performance: Summarise revenue, orders, AOV, and margin for the reporting month
versus the prior 3-month average. Be specific with numbers and directional language.

Paragraph 2 — What is driving the results: Explain the customer mix (new vs. repeat buyers) and
traffic source trends that are behind the top-line movement. Call out the most significant factors.

Paragraph 3 — Risks and areas to watch: Identify any metrics that are flat, declining, or warrant
monitoring. Suggest one or two questions the business should investigate next.

--- DATA ---

Reporting month: {current['month']}

Top-line KPIs (vs. prior 3-month average):
  Revenue    : ${current['revenue']:,.0f}  ({pct_change(current['revenue'], prior_avg['revenue'])})
  Orders     : {current['orders']:,.0f}  ({pct_change(current['orders'], prior_avg['orders'])})
  AOV        : ${current['aov']:,.2f}  ({pct_change(current['aov'], prior_avg['aov'])})
  Margin %   : {current['margin_pct']:.1f}%  ({pct_change(current['margin_pct'], prior_avg['margin_pct'])})
  Return rate: {current['return_rate']:.1f}%  ({pct_change(current['return_rate'], prior_avg['return_rate'])})

Customer order mix ({current['month']}):
{cx_lines}

Session volume by traffic source ({current['month']}):
{traffic_lines}
"""

print(prompt)

You are a senior data analyst writing a monthly executive summary for a retail e-commerce business.
Write exactly three paragraphs. Do not use bullet points or headers — prose only.

Paragraph 1 — Top-line performance: Summarise revenue, orders, AOV, and margin for the reporting month
versus the prior 3-month average. Be specific with numbers and directional language.

Paragraph 2 — What is driving the results: Explain the customer mix (new vs. repeat buyers) and
traffic source trends that are behind the top-line movement. Call out the most significant factors.

Paragraph 3 — Risks and areas to watch: Identify any metrics that are flat, declining, or warrant
monitoring. Suggest one or two questions the business should investigate next.

--- DATA ---

Reporting month: 2026-03

Top-line KPIs (vs. prior 3-month average):
  Revenue    : $204,278  (+89.8%)
  Orders     : 2,354  (+88.2%)
  AOV        : $86.78  (+0.9%)
  Margin %   : 51.2%  (-0.1%)
  Return rate: 0.0%  (N/A)

Customer order m

## 6. Generate the Summary — Claude API

In [9]:
response = claude_client.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=600,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

summary = response.content[0].text

## 7. Output

In [10]:
print('=' * 70)
print(f'  EXECUTIVE SUMMARY — {current["month"]}')
print(f'  Generated {datetime.now().strftime("%Y-%m-%d %H:%M")} via {CLAUDE_MODEL}')
print('=' * 70)
print()
print(summary)
print()
print('=' * 70)
print(f'  Input tokens : {response.usage.input_tokens}')
print(f'  Output tokens: {response.usage.output_tokens}')
print('=' * 70)

  EXECUTIVE SUMMARY — 2026-03
  Generated 2026-04-06 11:55 via claude-haiku-4-5-20251001

# Executive Summary – March 2026

March delivered exceptional top-line growth, with revenue reaching $204,278, up 89.8% against the prior three-month average, and orders climbing 88.2% to 2,354 units. This near-doubling of volume was achieved while holding average order value relatively flat at $86.78, representing a modest 0.9% uplift, suggesting that growth was primarily driven by transaction count rather than customer spending increases. Gross margin remained stable at 51.2%, declining only 0.1 percentage points year-over-year, which indicates we successfully scaled operations without material pressure on profitability. Notably, the return rate recorded zero returns during the period, though this metric warrants validation given the atypical result.

The surge in performance was fueled by a balanced contribution across new and repeat customer segments. New customers generated 1,295 orders worth